# Entornos de Transformación Digital: Ingeniería y Selección de Características, Optimización y Modelado Estratégico con Scikit-Learn
---

### Propósito de la Sesión y Visión de Alta Gerencia
En los procesos de transformación digital, los datos operacionales rara vez están listos para alimentar directamente un motor de inteligencia artificial. La **Ingeniería de Características (*Feature Engineering*)** constituye el puente estratégico entre los sistemas transaccionales del negocio y los modelos predictivos.

Para la alta gerencia, este proceso responde a tres imperativos de gobierno y rentabilidad:
1. **Eficiencia y Reducción de Costos (*Compute & Storage*):** Disminuir la dimensionalidad reduce tiempos de procesamiento, consumo de infraestructura en la nube y costos operativos de inferencia.
2. **Explicabilidad y Mitigación de Ruido:** Eliminar variables redundantes o no informativas mitiga el riesgo de falsos patrones y facilita la auditoría ante comités regulatorios y directivos.
3. **Agilidad en la Toma de Decisiones:** Transformar variables complejas en representaciones condensadas que capturan la máxima variabilidad operativa de la empresa.

## 1. Adquisición y Gobernanza de Datos (*Data Ingestion*)
> Cargamos un conjunto estructurado que simula variables cuantitativas de un ecosistema organizacional para resolver un problema arquetípico de categorización y toma de decisiones.

In [15]:
# Importación de bibliotecas analíticas base
import numpy as np
import pandas as pd
from sklearn import datasets

# Ingesta estructurada como DataFrame de Pandas
iris = datasets.load_iris(as_frame=True)
X = iris.data
y = iris.target

df_resumen = iris.frame.copy()
df_resumen['clase_objetivo'] = iris.target_names[iris.target]
df_resumen.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,clase_objetivo
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


## 2. Partición Estratégica: Entrenamiento vs. Validación (*Holdout Split*)
> Separamos el 20% de los datos para la evaluación final no observada (*Holdout*), garantizando que ni la selección de variables ni la calibración del modelo sufran de fuga de información (*Data Leakage*).

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Casos para ingeniería y optimización (Train): {X_train.shape[0]}")
print(f"Casos reservados para validación de negocio (Test): {X_test.shape[0]}")

Casos para ingeniería y optimización (Train): 120
Casos reservados para validación de negocio (Test): 30


## 3. Ingeniería y Reducción de Características (*Feature Engineering*)

### Enfoque Estratégico para Líderes de Transformación Digital
La ingeniería de características abarca dos grandes vertientes complementarias:

- **Selección Automática de Características (*Feature Selection*):** Filtra y conserva únicamente aquellas variables de negocio existentes que mayor poder predictivo aportan a la variable objetivo, descartando indicadores con nulo aporte estadístico.
- **Extracción / Reducción de Dimensionalidad con PCA (*Principal Component Analysis*):** Proyecta y sintetiza múltiples indicadores intercorrelacionados en nuevos ejes abstractos ortogonales (componentes principales) que retienen la máxima varianza de la organización, mitigando la redundancia.

### 3.1. Estandarización de Variables (*Data Preprocessing*)

Tanto los métodos de selección como las técnicas de reducción dimensional basadas en varianza (como PCA) requieren que los datos operacionales compartan la misma escala (media 0, desviación estándar 1). Esto evita que indicadores con cifras monetarias o magnitudes elevadas dominen indebidamente el análisis.

In [17]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X_train_scaled = escalador.fit_transform(X_train)
X_test_scaled = escalador.transform(X_test)

print("Variables estandarizadas con éxito para calibración analítica.")

Variables estandarizadas con éxito para calibración analítica.


### 3.2. Selección Automática de Características: `SelectKBest`
> Supongamos que por restricciones de costo o telemetría en sistemas transaccionales, solo podemos monitorear las 2 variables más determinantes.
Utilizamos `SelectKBest` con la prueba ANOVA F-value (`f_classif`) para clasificar el poder de discriminación de cada variable respecto a la meta organizacional.

In [ ]:
from IPython.display import display
from sklearn.feature_selection import SelectKBest, f_classif

# Configuración del selector para conservar las mejores 2 características de negocio
selector = SelectKBest(score_func=f_classif, k=2)
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest = selector.transform(X_test_scaled)

# Diagnóstico gerencial de relevancia de variables
df_importancia = pd.DataFrame({
    'Variable': X.columns,
    'Puntuacion_F': selector.scores_,
    'Seleccionada': selector.get_support()
}).sort_values(by='Puntuacion_F', ascending=False)

print("Ranking de Relevancia de Variables:")
display(df_importancia)

Ranking de Relevancia de Variables:


,Variable,Puntuacion_F,Seleccionada
2,petal length (cm),948.890381,True
3,petal width (cm),803.214088,True
0,sepal length (cm),100.965923,False
1,sepal width (cm),36.030577,False


### 3.3. Reducción de Dimensionalidad con PCA (*Principal Component Analysis*)
> En lugar de descartar variables, `PCA` combina la información dispersa de todos los indicadores en 2 macroindicadores sintéticos (componentes principales), eliminando la multicolinealidad y facilitando el monitoreo gerencial en tableros ejecutivos 2D.

In [18]:
from sklearn.decomposition import PCA

# Proyección a 2 componentes principales ortogonales
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

varianza_explicada = pca.explained_variance_ratio_
print(f"Varianza retenida por Componente 1: {varianza_explicada[0]:.2%}")
print(f"Varianza retenida por Componente 2: {varianza_explicada[1]:.2%}")
print(f"Información total capturada del negocio: {varianza_explicada.sum():.2%}")

# Matriz de pesos (Cargas) de cada variable original en los macroindicadores PCA
df_pca_cargas = pd.DataFrame(
    pca.components_,
    columns=X.columns,
    index=['Macro_Componente_1', 'Macro_Componente_2']
)
print("\nPesos de las variables de negocio en cada macrocomponente:")
display(df_pca_cargas)

Varianza retenida por Componente 1: 72.68%
Varianza retenida por Componente 2: 23.07%
Información total capturada del negocio: 95.74%

Pesos de las variables de negocio en cada macrocomponente:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
Macro_Componente_1,0.526793,-0.253072,0.581869,0.565572
Macro_Componente_2,0.348139,0.934708,0.026894,0.066308


## 4. Evaluación Robusta del Rendimiento: Validación Cruzada (*Cross-Validation*)

Evaluamos la estabilidad del modelo sobre los datos transformados mediante **Validación Cruzada Estratificada de 5 Pliegues**, mitigando sesgos de estimación.

In [19]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
modelo_base = KNeighborsClassifier(n_neighbors=5)

# Evaluación sobre el espacio reducido por PCA
scores_cv = cross_val_score(modelo_base, X_train_pca, y_train, cv=cv_estratificado, scoring='accuracy')

print(f"Rendimientos por pliegue (CV 5-Fold con PCA): {np.round(scores_cv, 4)}")
print(f"Exactitud media esperada: {scores_cv.mean():.2%}")
print(f"Estabilidad del modelo (Desviación Estándar): ±{scores_cv.std():.4f}")

Rendimientos por pliegue (CV 5-Fold con PCA): [0.875  0.9167 0.875  0.9583 0.9167]
Exactitud media esperada: 90.83%
Estabilidad del modelo (Desviación Estándar): ±0.0312


## 5. Optimización Exhaustiva de Hiperparámetros: Grid Search CV

Exploramos exhaustivamente el espacio de hiperparámetros sobre los macrocomponentes generados, garantizando la configuración de mayor precisión.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=cv_estratificado,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_pca, y_train)

print("Resultados Grid Search CV ")
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor exactitud validada en CV: {grid_search.best_score_:.2%}")

=== Resultados Grid Search CV 
Mejores hiperparámetros: {'metric': 'manhattan', 'n_neighbors': 7, 'weights': 'uniform'}
Mejor exactitud validada en CV: 91.67%


## 6. Optimización Eficiente en Espacios Complejos: Randomized Search CV

Estrategia ágil para proyectos con restricción de tiempo computacional, explorando distribuciones probabilísticas de parámetros mediante un número acotado de iteraciones (`n_iter=15`).

In [22]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_neighbors': randint(1, 30),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

random_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=15,
    cv=cv_estratificado,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_pca, y_train)

print("Resultados Randomized Search CV")
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor exactitud validada en CV: {random_search.best_score_:.2%}")

Resultados Randomized Search CV
Mejores hiperparámetros: {'n_neighbors': 7, 'p': 2, 'weights': 'uniform'}
Mejor exactitud validada en CV: 90.00%


## 7. Despliegue y Evaluación Ejecutiva del Mejor Modelo

Auditamos el mejor modelo obtenido contra los datos de prueba (`X_test_pca`), asegurando que las decisiones automatizadas mantengan altos estándares de precisión, exhaustividad y balance.

In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

mejor_modelo = grid_search.best_estimator_
y_pred_final = mejor_modelo.predict(X_test_pca)

exactitud_final = accuracy_score(y_test, y_pred_final)
print(f"Exactitud en producción simulada (Test Set): {exactitud_final:.2%}\n")
print("Reporte Ejecutivo de Clasificación:")
print(classification_report(y_test, y_pred_final, target_names=iris.target_names))

print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred_final))

Exactitud en producción simulada (Test Set): 90.00%

Reporte Ejecutivo de Clasificación:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Matriz de Confusión:
[[10  0  0]
 [ 0  9  1]
 [ 0  2  8]]


In [25]:
import warnings
warnings.filterwarnings('ignore')

## 8. Inferencia Predictiva Automatizada sobre Casos de Negocio

Aplicamos el pipeline completo (Estandarización $\rightarrow$ Proyección PCA $\rightarrow$ Inferencia de Clasificación) ante nuevos eventos capturados por los sistemas de la empresa.

In [26]:
# Nuevos eventos del entorno operativo (4 variables originales)
nuevos_eventos = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.2, 2.8, 4.8, 1.8],
    [7.3, 3.0, 6.3, 1.8]
])

# Pipeline de inferencia: Escalamiento -> PCA -> Modelo
nuevos_escalados = escalador.transform(nuevos_eventos)
nuevos_pca = pca.transform(nuevos_escalados)

predicciones = mejor_modelo.predict(nuevos_pca)
probabilidades = mejor_modelo.predict_proba(nuevos_pca)

for i, (evento, pred, prob) in enumerate(zip(nuevos_eventos, predicciones, probabilidades), start=1):
    categoria = iris.target_names[pred]
    confianza = np.max(prob)
    print(f"Evento Operativo {i}: Clasificado como '{categoria}' con certeza del {confianza:.2%}")

Evento Operativo 1: Clasificado como 'setosa' con certeza del 100.00%
Evento Operativo 2: Clasificado como 'virginica' con certeza del 71.43%
Evento Operativo 3: Clasificado como 'virginica' con certeza del 100.00%
